In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import os



from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.abspath(".."))
from core.viz import (plot_bar, plot_dynamic_trends, plot_line,
 plot_corr_triangle, plot_scatter, plot_statistical_strip, plot_heatmap, 
create_subplot_grid)
from core.s3 import S3AssetManager

In [2]:

s3 = S3AssetManager(notebook_name="nutrinor_performance_trimestre_2025")


In [3]:


def decile_summary(
    df: pd.DataFrame,
    col: str,
    q: int = 20,
    decile_col: str | None = None,
    dropna_for_summary: bool = True,
) -> pd.DataFrame:
    """
    Crea una columna de cuantiles (por defecto deciles) y devuelve un resumen por grupo.
    - Maneja nulos (quedan NaN en la columna de decil).
    - Maneja empates (duplicates='drop').

    Returns: DataFrame con ['decil', 'n', 'mean', 'p50', 'min', 'max'].
    """
    if decile_col is None:
        decile_col = f"q{q}_{col}"

    x = pd.to_numeric(df[col], errors="coerce")

    # asignación de cuantiles (1..K). NaNs se mantienen como NaN
    df[decile_col] = pd.qcut(x, q=q, labels=False, duplicates="drop")
    df[decile_col] = df[decile_col].astype("float") + 1  # float para permitir NaN

    g = df if not dropna_for_summary else df[df[decile_col].notna()]

    summary = (g.groupby(decile_col)[col]
               .agg(n="count", mean="mean", p50="median", min="min", max="max")
               .reset_index()
               .rename(columns={decile_col: "decil"}))

    return summary




def categorizar_especie_detallada(nombre_producto):
    # Estandarizar a mayúsculas
    nombre = str(nombre_producto).upper()
    
    # Diccionarios de palabras clave
    especies = {
        'Aves': ['POLLO', 'GALLINA', 'PONEDORA', 'ENGORDE GM', 'ENGORDE BR', 'POLLITO', 'POLLITO'],
        'Mascotas': ['PERRO', 'CANINO', 'GATO', 'FELINO', 'ADULTO', 'CACHORRO'],
        'Caballos': ['EQUINOS', 'CABALLO', 'YEGUA', 'POTRO'],
        'Ganadería': [
            'LECHE', 'GANADERIA', 'ENERGIA', 'TERNERAS', 'LECHER', 'VACUNACION', 
            'PREPARTO', 'FIBROSO', 'NOVILLO', 'SALAS LB', '75-16', '75-18', 
            'ALTO DESEMPEÑO', 'MARR-PREPTO' # 'MARR-PREPTO' detectado como Ganadería en tu imagen
        ],
        'Porcicultura': [
            'MARRANA', 'LECHON', 'CERDO', 'LECHONES', 'CEBA', 'INICIACION', 
            'FINALIZADOR', 'LEVANTE', 'DESARROLLO', 'GESTACION', 'MAGRO', 
            'CRECIMIENTO', 'REEMPLAZO', 'PREINICIADOR', 'PRELACTANCIA', 'PRELEVANTE',

            #TODO
            'DESEMPEÑO NORTE SOLIDO PELET', 'ENGORDE HD',
       'DESEMPEÑO NORTE PELET', 'ENGORDE HD DOXI'
        ]
    }

    # --- LÓGICA DE PRIORIDAD ---
    
    # 1. Primero buscamos si el nombre tiene términos de Porcicultura o Ganadería
    # Esto asegura que "MAQUILA PREINICIADOR" sea Porcicultura
    if any(key in nombre for key in especies['Porcicultura']):
        return 'Porcicultura'
        
    if any(key in nombre for key in especies['Ganadería']):
        return 'Ganadería'

    # 2. Luego revisamos las especies más pequeñas
    for esp in ['Aves', 'Mascotas', 'Caballos']:
        if any(key in nombre for key in especies[esp]):
            return esp
    
    # 3. Si no encontró ninguna especie pero dice MAQUILA, se queda como Maquila general
    if 'MAQUILA' in nombre:
        return 'Maquila'
        
    # 4. Caso por defecto
    return 'Por_definir'

# Aplicar a tu columna
# df['especie'] = df['product'].apply(categorizar_especie_final)





In [4]:
path = '../raw/Rendimiento pelletizadoras 2024 (2).xlsx'

pellet420 = pd.read_excel(path,
skiprows=1)

pellet420["pellet"] = 'pellet 420'

pellet520 = pd.read_excel(path, 
sheet_name="rendimiento pellet 2 (520)",
skiprows=1) 

pellet520["pellet"] = 'pellet 520'

pellet350 = pd.read_excel(path, 
sheet_name="rendimiento pellet 3 (350)",
skiprows=1)

pellet350["pellet"] = 'pellet 350'

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [5]:
df = pd.concat([pellet420, pellet520, pellet350], ignore_index=True)

In [6]:
rename_map = {
    #"año": "year",
    #"mes": "month",
    #"semana": "week",
    "Fecha": "date",
    "Producto": "product",
    "Pelletizador": "operator",
    "pellet": "pellet",
    "Hora inicial ": "start_time",
    "Hora final ": "end_time",
    "tiempo trabajado (horas)": "worked_time_hours",
    "bultos": "bags",
    "durabilidad (%)": "durability_pct",
    "carga (Hz)": "load_hz",
    "toneladas": "tons",
    "ton/hora": "tph",
    "lote": "batch",
    "Dado": "die",
    "ROLLER": "roller",
    "OBSERVACIONES": "notes",

}

df = df.rename(columns=rename_map)


In [7]:
df_dep = df[rename_map.values()]

dt = pd.to_datetime(df_dep["date"], errors="coerce")
df_dep = df_dep.loc[dt.notna()].copy()
dt = dt.loc[dt.notna()]
df_dep["year_month"] = dt.dt.to_period("M").astype(str)
df_dep = df_dep.sort_values(by="date", ascending=False)

df_dep['especie'] = df_dep['product'].apply(categorizar_especie_detallada)
df_dep = df_dep[df_dep["date"]>='2024-05-01']

df_dep['periodo'] = np.where(df_dep['year_month'] < '2025-04-01', 'Pre', 'Post')

In [8]:
cls_num = ['worked_time_hours', 'bags', 'durability_pct', 'load_hz', 'tons', 'batch', 'die', 'tph']
for cl in cls_num:
    df_dep[cl] = pd.to_numeric(df_dep[cl], errors='coerce')
df_dep = df_dep[df_dep["batch"].notnull()]
#df_dep["tph"] = df_dep["tons"]/df_dep['worked_time_hours'].copy()

df_dep["corrida"] =  np.where(df_dep["tons"]<=10, 'corto', 'largo')

In [9]:
decile_summary(df_dep, "durability_pct")

,decil,n,mean,p50,min,max
0,1.0,795,93.404528,95.0,5.60,95.0
1,2.0,471,95.200000,95.2,95.20,95.2
2,3.0,1289,95.591078,95.6,95.40,95.6
3,4.0,1627,95.995390,96.0,95.70,96.0
4,5.0,381,96.200000,96.2,96.20,96.2
5,6.0,810,96.399765,96.4,96.21,96.4
6,7.0,171,96.599415,96.6,96.50,96.6
7,8.0,856,96.799579,96.8,96.70,96.8
8,9.0,829,97.147889,97.2,96.90,97.2
9,10.0,494,97.589109,97.6,97.40,97.6


In [10]:
df_dep.loc[(df_dep['durability_pct']< 94) | (df_dep['durability_pct']> 100), "durability_pct"] = df_dep["durability_pct"].median()

In [11]:
decile_summary(df_dep, "load_hz")

,decil,n,mean,p50,min,max
0,1.0,604,13.120364,14.0,0.0,14.0
1,2.0,535,14.991589,15.0,14.5,15.0
2,3.0,1254,15.996651,16.0,15.4,16.0
3,4.0,828,16.976449,17.0,16.2,17.0
4,5.0,719,17.996523,18.0,17.4,18.0
5,6.0,998,19.733567,20.0,18.5,20.0
6,7.0,53,21.000000,21.0,21.0,21.0
7,8.0,405,22.580247,22.0,22.0,24.0
8,9.0,365,25.000000,25.0,25.0,25.0
9,10.0,652,26.420245,26.0,26.0,27.0


In [12]:
df_dep.loc[(df_dep['load_hz']< 10) | (df_dep['load_hz']> 40), "load_hz"] = df_dep["load_hz"].median()

In [13]:
df_dep.loc[df_dep['year_month'] == '2026-12', "year_month"] = '2025-12'

In [14]:
def build_group(df, group):

    df_group = df.groupby(group).agg(
    tons=("tons", "sum"),
    hours=("worked_time_hours", "sum"),
    durability_pct=("durability_pct", "mean"),
    load_hz=("load_hz", "mean"),
    tph=("tph", "mean")).reset_index()
    return df_group

In [15]:
by_month = build_group(df_dep, 'year_month')
by_month

,year_month,tons,hours,durability_pct,load_hz,tph
0,2024-05,6873.040,1101.750000,96.619802,21.933920,6.313142
1,2024-06,7200.260,1151.083333,96.754545,21.723492,6.462177
2,2024-07,6617.440,1043.000000,96.558612,22.953642,6.732572
3,2024-08,6621.680,988.416667,96.406182,23.009474,7.602800
4,2024-09,6648.872,1033.416667,96.595050,23.957447,6.735368
5,2024-10,7134.880,1126.083333,96.662996,24.451429,7.121201
6,2024-11,6470.880,975.333333,96.575324,23.491667,7.382090
7,2024-12,6454.320,897.833333,96.707765,27.407960,7.611777
8,2025-01,6761.160,915.250000,96.518351,28.919149,7.662578
9,2025-02,6394.240,884.750000,96.509268,26.370968,7.760675


In [47]:
# 1. Calcular el cambio porcentual con respecto al mes anterior
by_month['pct_change'] = by_month['tons'].pct_change() * 100

# 2. Definir flecha y color según el movimiento
def get_hover_info(pct):
    if pd.isna(pct): return ""
    icon = "▲" if pct > 0 else "▼"
    color = "green" if pct > 0 else "red"
    # Formateamos con HTML para que Plotly lo interprete en el hover
    return f"<span style='color:{color}'>{icon} {abs(pct):.1f}%</span>"

by_month['hover_pct'] = by_month['pct_change'].apply(get_hover_info)
# Generar la gráfica base con tu función
f = plot_line(by_month, x_col='year_month', y_col="tons", title='Producción en Planta',
             x_title='Mes', y_title="Producción(Ton)")

# Actualizar los datos del hover
# custom_data permite pasar columnas adicionales al objeto gráfico
f.update_traces(
    customdata=by_month['hover_pct'],
    hovertemplate=(
        "<b>Mes:</b> %{x}<br>" +
        "<b>Producción:</b> %{y:,.0f} Ton<br>" +
        "<b>Var. Mensual:</b> %{customdata}" +
        "<extra></extra>" # Esto elimina el nombre de la traza a la derecha
    )
)

f.show()
s3.save_plotly_html(f, "prod_planta.html")

In [48]:
by_month_pellet = build_group(df_dep, ['year_month', 'pellet'])

In [53]:
f_prod_pellet = plot_line(by_month_pellet, x_col='year_month', y_col="tons", group_col='pellet',
title='Producción por Pellet',
 x_title='Mes', 
 y_title="Producción(Ton)")
#f_prod_pellet.show()
s3.save_plotly_html(f_prod_pellet, 'prod_pellet.html')

f_carga_pellet = plot_line(by_month_pellet, x_col="year_month", y_col="load_hz", group_col="pellet",
 title='Carga por Pellet',
 x_title='Mes', 
 y_title="Carga (Hz)")

#f_carga_pellet.show()
s3.save_plotly_html(f_carga_pellet, 'carga_pellet.html')

f_tph_pellet = plot_line(by_month_pellet, x_col="year_month", y_col="tph", group_col="pellet",
 title='Rendimiento por Pellet',
 x_title='Mes', 
 y_title="Rendimiento (Ton/h)")
#f_tph_pellet.show()

s3.save_plotly_html(f_tph_pellet, 'tph_pellet.html')
f_pdi_pellet = plot_line(by_month_pellet, x_col="year_month", y_col="durability_pct", group_col="pellet",
 title='Durabilidad por Pellet',
 x_title='Mes', 
 y_title="Durabilidad (%)")
#f_pdi_pellet.show()

s3.save_plotly_html(f_pdi_pellet, "pdi_pellet.html")

f_group_pellet = create_subplot_grid(
    main_title="<b>Producción, Carga, TPH y PDI por Pellet </b>",
    figures = [f_prod_pellet, f_carga_pellet, f_tph_pellet, f_pdi_pellet],
    rows=2, cols=2,
    titles=["<b>Producción(Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=600,
    width=1200,
    )

f_group_pellet.show()
s3.save_plotly_html(f_group_pellet, "group_pellet.html")


In [82]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Paleta institucional
DEFAULT_COLORS = ["#1C8074", "#666666", "#FFA726", "#29B6F6", "#29B6F6",]

# Mapeo de colores fijo por pellet
unique_pellets = sorted(by_month_pellet['pellet'].unique())
color_palette_map = {pellet: DEFAULT_COLORS[i % len(DEFAULT_COLORS)] 
                     for i, pellet in enumerate(unique_pellets)}

# Preparación de indicadores visuales para el hover
def get_delta_label(pct):
    if pd.isna(pct): return "N/A"
    icon = "▲" if pct > 0 else "▼"
    color = "green" if pct > 0 else "red"
    return f"<span style='color:{color}'>{icon} {abs(pct):.1f}%</span>"

metrics_map = {'tons': 'hover_prod', 'load_hz': 'hover_carga', 
               'tph': 'hover_tph', 'durability_pct': 'hover_pdi'}

for col, hover_col in metrics_map.items():
    pct_change = by_month_pellet.groupby('pellet')[col].pct_change() * 100
    by_month_pellet[hover_col] = pct_change.apply(get_delta_label)

by_month_pellet['date_str'] = by_month_pellet['year_month'].astype(str)

In [83]:
def create_custom_line(df, y_col, hover_col, title, y_label):
    fig = px.line(
        df, x='year_month', y=y_col, color='pellet',
        color_discrete_map=color_palette_map,
        custom_data=['date_str', hover_col]
    )
    
    # Engrosamos la línea y ajustamos el marcador
    fig.update_traces(
        mode='lines+markers',
        line=dict(width=4),      # <--- Grosor de línea aumentado
        marker=dict(size=8, line=dict(width=1, color='white')), # Marcadores más definidos
        hovertemplate="<b>Mes:</b> %{customdata[0]}<br>" +
                      "<b>Valor:</b> %{y:,.1f}<br>" +
                      "<b>Var. Mensual:</b> %{customdata[1]}<extra></extra>"
    )
    
    fig.update_layout(
        xaxis_title="Mes", 
        yaxis_title=y_label,
        plot_bgcolor='white',
        hovermode='x unified'
    )
    return fig

# Generar las 4 gráficas
f_prod_pellet = create_custom_line(by_month_pellet, 'tons', 'hover_prod', 'Producción', 'Tons')
f_carga_pellet = create_custom_line(by_month_pellet, 'load_hz', 'hover_carga', 'Carga', 'Hz')
f_tph_pellet = create_custom_line(by_month_pellet, 'tph', 'hover_tph', 'Rendimiento', 'Ton/h')
f_pdi_pellet = create_custom_line(by_month_pellet, 'durability_pct', 'hover_pdi', 'PDI', '%')

In [87]:
f_group_pellet = create_subplot_grid(
    main_title="<b>Control Pelletizado</b><br></sup>",
    figures = [f_prod_pellet, f_carga_pellet, f_tph_pellet, f_pdi_pellet],
    rows=2, cols=2,
    titles=["<b>Producción (Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=800,
    width=1300,
)

# Ajustes de Layout final
f_group_pellet.update_layout(
    legend_title_text='Línea de Pellet',
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
    margin=dict(t=100, b=100)
)

# Rejilla sutil para facilitar la lectura
f_group_pellet.update_xaxes(showgrid=True, gridcolor='#F2F2F2')
f_group_pellet.update_yaxes(showgrid=True, gridcolor='#F2F2F2')

f_group_pellet.show()

# Guardado
s3.save_plotly_html(f_group_pellet, "group_pellet.html")

In [89]:
by_month_pellet_especie = build_group(df_dep[df_dep["especie"].isin(['Porcicultura', 'Ganadería'])], ['year_month', 'pellet', 'especie'])
by_month_pellet_especie

,year_month,pellet,especie,tons,hours,durability_pct,load_hz,tph
0,2024-05,pellet 420,Ganadería,1875.52,343.083333,96.690625,31.336957,5.421933
1,2024-05,pellet 420,Porcicultura,909.00,183.166667,95.920000,26.558824,4.674928
2,2024-05,pellet 520,Ganadería,212.00,28.250000,96.837500,19.975000,7.668560
3,2024-05,pellet 520,Porcicultura,3617.52,501.333333,96.922319,16.310500,7.197013
4,2024-06,pellet 420,Ganadería,2115.76,423.416667,96.809901,27.892473,4.950773
...,...,...,...,...,...,...,...,...
99,2026-01,pellet 350,Porcicultura,1145.44,419.000000,96.570079,19.104000,2.710259
100,2026-01,pellet 420,Ganadería,2075.72,447.166667,96.931313,20.050000,4.675459
101,2026-01,pellet 420,Porcicultura,288.00,70.583333,95.980000,16.600000,4.077972
102,2026-01,pellet 520,Ganadería,240.00,29.166667,96.573333,21.328571,8.362029


In [90]:
f_pdi_pellet420_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 420"], 
    x_col="year_month"
    , y_col="durability_pct",
     group_col="especie",
 title='Durabilidad por Especie en pellet 420',
 x_title='Mes', 
 y_title="Durabilidad (%)")
f_pdi_pellet420_especie.show()

s3.save_plotly_html(f_pdi_pellet420_especie, "f_pdi_pellet420_especie.html")

In [91]:
f_prod_pellet420_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 420"], 
    x_col="year_month"
    , y_col="tons",
     group_col="especie",
 title='Producción por Especie en pellet 420',
 x_title='Mes', 
 y_title="Producción (ton)")
f_prod_pellet420_especie.show()

s3.save_plotly_html(f_prod_pellet420_especie, "f_prod_pellet420_especie.html")

In [92]:
f_carga_pellet420_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 420"], 
    x_col="year_month"
    , y_col="load_hz",
     group_col="especie",
 title='Carga por Especie en pellet 420',
 x_title='Mes', 
 y_title="Carga (hz)")
f_carga_pellet420_especie.show()

s3.save_plotly_html(f_carga_pellet420_especie, "f_carga_pellet420_especie.html")

In [93]:
f_tph_pellet420_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 420"], 
    x_col="year_month"
    , y_col="tph",
     group_col="especie",
 title='Rendimiento por Especie en pellet 420',
 x_title='Mes', 
 y_title="Tph")
f_tph_pellet420_especie.show()

s3.save_plotly_html(f_tph_pellet420_especie, "f_tph_pellet420_especie.html")

In [94]:
f_group_pellet = create_subplot_grid(
    main_title="<b>Producción, Carga, TPH y PDI por Especie en pellet 420 </b>",
    figures = [f_prod_pellet420_especie, f_carga_pellet420_especie, f_tph_pellet420_especie, f_pdi_pellet420_especie],
    rows=2, cols=2,
    titles=["<b>Producción(Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=600,
    width=1200,
    )

f_group_pellet.show()
s3.save_plotly_html(f_group_pellet, "group_especie_pellet420.html")

In [97]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
by_month_pellet420_especie = by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 420"]
# 1. Definir paleta y mapeo de colores por especie
DEFAULT_COLORS = ["#1C8074", "#666666", "#E4572E", "#29B6F6", "#FFA726"]
unique_especies = sorted(by_month_pellet420_especie['especie'].unique())
color_palette_especie = {esp: DEFAULT_COLORS[i % len(DEFAULT_COLORS)] 
                         for i, esp in enumerate(unique_especies)}

# 2. Función de indicadores visuales (Flechas)
def get_delta_label(pct):
    if pd.isna(pct): return "N/A"
    icon = "▲" if pct > 0 else "▼"
    color = "green" if pct > 0 else "red"
    return f"<span style='color:{color}'>{icon} {abs(pct):.1f}%</span>"

# 3. Cálculo de variaciones por métrica y especie
metrics_map = {'tons': 'hover_prod', 'load_hz': 'hover_carga', 
               'tph': 'hover_tph', 'durability_pct': 'hover_pdi'}

for col, hover_col in metrics_map.items():
    # Calculamos pct_change agrupado por ESPECIE
    
    pct_change = by_month_pellet420_especie.groupby('especie')[col].pct_change() * 100
    by_month_pellet420_especie[hover_col] = pct_change.apply(get_delta_label)

by_month_pellet420_especie['date_str'] = by_month_pellet420_especie['year_month'].astype(str)

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/1453699446.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/1453699446.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/1453699446.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [98]:
def create_species_line(df, y_col, hover_col, title, y_label):
    fig = px.line(
        df, x='year_month', y=y_col, color='especie',
        color_discrete_map=color_palette_especie,
        custom_data=['date_str', hover_col]
    )
    
    fig.update_traces(
        mode='lines+markers',
        line=dict(width=4), # Líneas gruesas
        marker=dict(size=8, line=dict(width=1, color='white')),
        hovertemplate="<b>Mes:</b> %{customdata[0]}<br>" +
                      "<b>Valor:</b> %{y:,.1f}<br>" +
                      "<b>Var. Mensual:</b> %{customdata[1]}<extra></extra>"
    )
    
    fig.update_layout(xaxis_title="Mes", yaxis_title=y_label, plot_bgcolor='white')
    return fig

# Creamos las figuras individuales
f_prod_especie = create_species_line(by_month_pellet420_especie, 'tons', 'hover_prod', 'Producción', 'Tons')
f_carga_especie = create_species_line(by_month_pellet420_especie, 'load_hz', 'hover_carga', 'Carga', 'Hz')
f_tph_especie = create_species_line(by_month_pellet420_especie, 'tph', 'hover_tph', 'Rendimiento', 'Ton/h')
f_pdi_especie = create_species_line(by_month_pellet420_especie, 'durability_pct', 'hover_pdi', 'PDI', '%')

In [105]:
f_group_pellet = create_subplot_grid(
    main_title="<b>Análisis por Especie: Producción y Calidad en Pellet 420</b><br>",
    figures = [f_prod_especie, f_carga_especie, f_tph_especie, f_pdi_especie],
    rows=2, cols=2,
    titles=["<b>Producción (Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=750,
    width=1200,
)

# Ajuste estético final
f_group_pellet.update_layout(
    legend_title_text='Especie',
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
    margin=dict(t=100, b=100)
)

# Rejilla sutil para lectura profesional
f_group_pellet.update_xaxes(showgrid=True, gridcolor='#F2F2F2')
f_group_pellet.update_yaxes(showgrid=True, gridcolor='#F2F2F2')

f_group_pellet.show()

# Guardar
s3.save_plotly_html(f_group_pellet, "group_especie_pellet420_premium.html")

In [106]:
f_prod_pellet520_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 520"], 
    x_col="year_month"
    , y_col="tons",
     group_col="especie",
 title='Producción por Especie en p ellet 520',
 x_title='Mes', 
 y_title="Producción (ton)")
f_prod_pellet520_especie.show()

s3.save_plotly_html(f_prod_pellet520_especie, "f_prod_pellet520_especie.html")


f_pdi_pellet520_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 520"], 
    x_col="year_month"
    , y_col="durability_pct",
     group_col="especie",
 title='Producción por Especie en pellet 5  20',
 x_title='Mes', 
 y_title="Durabilidad (%)")
f_pdi_pellet520_especie.show()

s3.save_plotly_html(f_pdi_pellet520_especie, "f_pdi_pellet520_especie.html")


f_carga_pellet520_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 520"], 
    x_col="year_month"  
    , y_col="load_hz",
     group_col="especie",
 title='Carga por Especie en pellet 520',
 x_title='Mes', 
 y_title="Carga (hz)")
f_carga_pellet520_especie.show()

s3.save_plotly_html(f_carga_pellet520_especie, "f_carga_pellet520_especie.html")


f_tph_pellet520_especie = plot_line(
    by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 520"], 
    x_col="year_month"
    , y_col="tph",
     group_col="especie",
 title='Rendimiento por Especie en pellet 520',
 x_title='Mes', 
 y_title="Tph")
f_tph_pellet520_especie.show()

s3.save_plotly_html(f_tph_pellet520_especie, "f_tph_pellet520_especie.html")


In [107]:
f_group_pellet = create_subplot_grid(
    main_title="<b>Producción, Carga, TPH y PDI por Especie en pellet 520 </b>",
    figures = [f_prod_pellet520_especie, f_carga_pellet520_especie, f_tph_pellet520_especie, f_pdi_pellet520_especie],
    rows=2, cols=2,
    titles=["<b>Producción(Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=600,
    width=1200,
    )

f_group_pellet.show()
s3.save_plotly_html(f_group_pellet, "group_especie_pellet520.html")

In [108]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Definir paleta institucional y mapeo para Especies
DEFAULT_COLORS = ["#1C8074", "#666666", "#E4572E", "#29B6F6", "#FFA726"]
by_month_pellet520_especie = by_month_pellet_especie[by_month_pellet_especie["pellet"] == "pellet 520"]
unique_especies_520 = sorted(by_month_pellet520_especie['especie'].unique())
color_palette_520 = {esp: DEFAULT_COLORS[i % len(DEFAULT_COLORS)] 
                     for i, esp in enumerate(unique_especies_520)}

# 2. Función de indicadores visuales (Flechas de Color)
def get_delta_label(pct):
    if pd.isna(pct): return "N/A"
    icon = "▲" if pct > 0 else "▼"
    color = "green" if pct > 0 else "red"
    return f"<span style='color:{color}'>{icon} {abs(pct):.1f}%</span>"

# 3. Cálculo de variaciones por métrica agrupado por especie
metrics_map = {'tons': 'hover_prod', 'load_hz': 'hover_carga', 
               'tph': 'hover_tph', 'durability_pct': 'hover_pdi'}

for col, hover_col in metrics_map.items():
    # Variación porcentual mensual
    pct_change = by_month_pellet520_especie.groupby('especie')[col].pct_change() * 100
    by_month_pellet520_especie[hover_col] = pct_change.apply(get_delta_label)

by_month_pellet520_especie['date_str'] = by_month_pellet520_especie['year_month'].astype(str)

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/3761596883.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/3761596883.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_81699/3761596883.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [109]:
def create_520_line(df, y_col, hover_col, title, y_label):
    fig = px.line(
        df, x='year_month', y=y_col, color='especie',
        color_discrete_map=color_palette_520,
        custom_data=['date_str', hover_col]
    )
    
    fig.update_traces(
        mode='lines+markers',
        line=dict(width=4), # Líneas gruesas para reporte ejecutivo
        marker=dict(size=8, line=dict(width=1, color='white')),
        hovertemplate="<b>Mes:</b> %{customdata[0]}<br>" +
                      "<b>Valor:</b> %{y:,.1f}<br>" +
                      "<b>Var. Mensual:</b> %{customdata[1]}<extra></extra>"
    )
    
    fig.update_layout(xaxis_title="Mes", yaxis_title=y_label, plot_bgcolor='white')
    return fig

# Creamos las figuras individuales para el Pellet 520
f_prod_520 = create_520_line(by_month_pellet520_especie, 'tons', 'hover_prod', 'Producción', 'Tons')
f_carga_520 = create_520_line(by_month_pellet520_especie, 'load_hz', 'hover_carga', 'Carga', 'Hz')
f_tph_520 = create_520_line(by_month_pellet520_especie, 'tph', 'hover_tph', 'Rendimiento', 'Ton/h')
f_pdi_520 = create_520_line(by_month_pellet520_especie, 'durability_pct', 'hover_pdi', 'PDI', '%')

In [110]:
f_group_pellet_520 = create_subplot_grid(
    main_title="<b>Análisis por Especie: Producción y Calidad en Pellet 520</b><br>",
    figures = [f_prod_520, f_carga_520, f_tph_520, f_pdi_520],
    rows=2, cols=2,
    titles=["<b>Producción (Tons)</b>", "<b>Carga (Hz)</b>", "<b>Rendimiento (Tons/Hora)</b>", "<b>PDI (%)</b>"],
    height=750,
    width=1200,
)

# Ajustes estéticos y de leyenda
f_group_pellet_520.update_layout(
    legend_title_text='Especie',
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
    margin=dict(t=100, b=100)
)

# Rejilla sutil para facilitar la lectura de coordenadas
f_group_pellet_520.update_xaxes(showgrid=True, gridcolor='#F2F2F2')
f_group_pellet_520.update_yaxes(showgrid=True, gridcolor='#F2F2F2')

f_group_pellet_520.show()

# Guardado del archivo HTML
s3.save_plotly_html(f_group_pellet_520, "group_especie_pellet520_premium.html")

In [33]:
cond_pel = df_dep["pellet"]=='pellet 420'
cond_dieta = df_dep["especie"].isin(["Ganadería", "Porcicultura"])
cond_date = df_dep["date"]>='2025-01-01'

df_dep[cond_pel & cond_dieta & cond_date].groupby(['die','roller', 'periodo']).agg(
durability_pct=('durability_pct', 'mean'),
tph=('tph', 'mean'),
load_hz=('load_hz', 'mean'),

).reset_index()

,die,roller,periodo,durability_pct,tph,load_hz
0,2.0,002-2,Post,95.583333,5.288169,21.021277
1,2.0,002-2,Pre,96.014286,5.649001,27.090909
2,3.0,003-2,Pre,97.184848,5.030812,24.954545
3,4.0,004-4,Post,96.818000,5.816294,28.370000
4,4.0,004-4,Pre,97.262570,6.080165,30.154286
5,5.0,005-5,Post,96.475926,5.542705,25.124424
6,5.0,005-6,Post,96.556522,4.403207,18.525000
7,6.0,006-7,Post,96.375849,5.216452,20.966292
8,6.0,006-8,Post,96.334694,5.161118,20.285714
9,6.0,006-9,Post,96.586777,4.928406,21.641667


In [34]:
df_dep[cond_pel & cond_dieta & cond_date].groupby(['especie', 'periodo']).agg(
durability_pct=('durability_pct', 'mean'),
tph=('tph', 'mean'),
load_hz=('load_hz', 'mean'),

).reset_index()

,especie,periodo,durability_pct,tph,load_hz
0,Ganadería,Post,96.626187,5.012336,21.830243
1,Ganadería,Pre,97.125636,5.613027,31.354212
2,Porcicultura,Post,95.726829,5.279980,21.455285
3,Porcicultura,Pre,96.240971,5.364419,28.124390


In [35]:

df_resumen = df_dep[cond_pel & cond_dieta & cond_date].groupby(['product', 'periodo']).agg({
    'durability_pct': 'mean',
    'load_hz': 'mean',
    'tons': 'sum',
    'tph': 'mean'
}).reset_index()

# 2. Pivotar para tener columnas 'Pre' y 'Post' lado a lado
df_pivot = df_resumen.pivot(index='product', columns='periodo', values='durability_pct')

# 3. Calcular el Delta (Caída)
# Positivo significa que bajó en el Post respecto al Pre
df_pivot['delta_pdi'] = df_pivot['Pre'] - df_pivot['Post']

# Filtrar solo los que cayeron (delta > 0) y tenían buen PDI inicial
promedio_pre = df_pivot['Pre'].mean()
productos_degradados = df_pivot[(df_pivot['delta_pdi'] > 0) & (df_pivot['Pre'] >= promedio_pre)]
productos_degradados

periodo,Post,Pre,delta_pdi
product,,,
75-16,96.978947,97.082353,0.103406
ALTA ENERGIA,96.640000,97.090323,0.450323
ALTA ENERGIA CP,96.718421,97.165854,0.447433
ALTA ENERGIA CP GRANEL,97.082353,97.208696,0.126343
ALTA ENERGIA GRANEL,96.941176,97.106667,0.165490
ALTA ENERGIA GV,96.530505,97.073171,0.542666
ALTA ENERGIA GV GRANEL,96.755000,97.104762,0.349762
ALTA ENERGIA R,96.550000,97.600000,1.050000
DESEMPEÑO NORTE PELET,96.135135,96.900000,0.764865


In [36]:

df_filtered = df_dep[cond_pel & cond_dieta & cond_date]

# A. Calcular Toneladas Promedio por Mes
# Primero sumamos toneladas por producto, periodo y mes
tons_por_mes = df_filtered.groupby(['product', 'periodo', 'year_month'])['tons'].sum().reset_index()

# Luego promediamos esos meses para obtener la "capacidad mensual" por periodo
avg_tons_mes = tons_por_mes.groupby(['product', 'periodo'])['tons'].mean().reset_index()
avg_tons_mes.columns = ['product', 'periodo', 'avg_monthly_tons']

# B. Calcular promedios de las otras variables (PDI, TPH, Load)
otras_metricas = df_filtered.groupby(['product', 'periodo']).agg({
    'durability_pct': 'mean',
    'tph': 'mean',
    'load_hz': 'mean'
}).reset_index()

# C. Unir todo y Pivotar
df_resumen = pd.merge(otras_metricas, avg_tons_mes, on=['product', 'periodo'])

# Pivotamos para tener Pre y Post en la misma fila
df_pivot = df_resumen.pivot(index='product', columns='periodo')
# Aplanar los nombres de las columnas (ej: 'durability_pct_Pre')
df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# 2. Cálculo de Diferencias (Deltas)
# delta < 0 significa que el valor bajó en el periodo Post

df_pivot['delta_pdi'] =  df_pivot['durability_pct_Post']- df_pivot['durability_pct_Pre']
df_pivot['delta_tons_mes'] = df_pivot['avg_monthly_tons_Post'] - df_pivot['avg_monthly_tons_Pre']
df_pivot['delta_tph'] = df_pivot['tph_Post']- df_pivot['tph_Pre']
df_pivot['delta_load'] = df_pivot['load_hz_Post'] - df_pivot['load_hz_Pre']

# 3. Filtrar Productos Degradados
# Que bajaron PDI y que inicialmente eran buenos (superior al promedio Pre)
# TODO: ¿Cuál seria el valor real que deberia ponerse?
promedio_pdi_pre = 90 #df_pivot['durability_pct_Pre'].mean()
#  & (df_pivot['durability_pct_Pre'] >= promedio_pdi_pre)

degradados = df_pivot[(df_pivot['delta_pdi'] < 0)].copy()

# 4. Ranking de Degradación
degradados['ranking_caida'] = degradados['delta_pdi'].rank(ascending=True)

degradados = degradados.sort_values(['ranking_caida'], ascending=True)
degradados

,product,durability_pct_Post,durability_pct_Pre,tph_Post,tph_Pre,load_hz_Post,load_hz_Pre,avg_monthly_tons_Post,avg_monthly_tons_Pre,delta_pdi,delta_tons_mes,delta_tph,delta_load,ranking_caida
82,PROBIOLECHE GRANEL,96.476033,97.527778,4.762748,5.506695,20.328333,29.760563,321.346667,335.50,-1.051745,-14.153333,-0.743947,-9.432230,1.0
12,ALTA ENERGIA R,96.550000,97.600000,5.382206,6.000000,26.000000,28.000000,10.000000,4.00,-1.050000,6.000000,-0.617794,-2.000000,2.0
28,DESEMPEÑO NORTE PELET,96.135135,96.900000,5.384408,5.944144,23.729730,32.050000,59.000000,75.00,-0.764865,-16.000000,-0.559736,-8.320270,3.0
81,PROBIOLECHE,96.445946,97.206250,5.397903,6.082675,22.094595,33.281250,206.222222,224.00,-0.760304,-17.777778,-0.684772,-11.186655,4.0
10,ALTA ENERGIA GV,96.530505,97.073171,5.149866,5.719172,23.415842,32.692308,267.480000,256.00,-0.542666,11.480000,-0.569306,-9.276466,5.0
46,LECHE TOP,96.551020,97.025000,5.255900,6.276824,22.720000,32.804348,74.666667,84.00,-0.473980,-9.333333,-1.020924,-10.084348,6.0
2,ALTA ENERGIA,96.640000,97.090323,5.381512,5.658286,23.953704,33.316667,326.506667,387.50,-0.450323,-60.993333,-0.276774,-9.362963,7.0
52,LEVANTE BR HORIZONTES,95.350000,95.800000,5.790664,6.428571,23.000000,26.000000,24.000000,7.00,-0.450000,17.000000,-0.637907,-3.000000,8.0
4,ALTA ENERGIA CP,96.718421,97.165854,5.415271,5.514377,23.050633,32.925000,195.333333,268.00,-0.447433,-72.666667,-0.099107,-9.874367,9.0
49,LECHONES,95.200000,95.620000,8.571429,5.677193,25.000000,29.500000,20.000000,20.00,-0.420000,0.000000,2.894236,-4.500000,10.0


In [111]:


class QualityPerformanceAnalyzer:
    """
    Analizador Estratégico de Calidad y Eficiencia Operativa.
    Diseñado para reportes ejecutivos de control de procesos industriales.
    """
    
    def __init__(self, dataframe, pellet_name=""):
        self.df = dataframe
        self.pellet_name = str(pellet_name).upper()
        self.processed_data = None
        self.results = None

    def transform_data(self, conds):
        """Etapa ETL: Normalización mensual y cálculo de variaciones (Post - Pre)."""
        # 1. Agregación mensual para normalizar variaciones de días laborados
        df_mensual = self.df[conds].groupby(['product', 'periodo', 'year_month']).agg({
            'tons': 'sum',
            'durability_pct': 'mean',
            'tph': 'mean',
            'load_hz': 'mean'
        }).reset_index()

        # 2. Promedio por periodo (Semestre)
        df_resumen = df_mensual.groupby(['product', 'periodo']).agg({
            'durability_pct': 'mean',
            'tons': 'mean',
            'tph': 'mean',
            'load_hz': 'mean'
        }).reset_index()
        df_resumen.rename(columns={'tons': 'avg_monthly_tons'}, inplace=True)

        # 3. Pivot para comparación directa (Post vs Pre)
        df_pivot = df_resumen.pivot(index='product', columns='periodo')
        df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns]
        df_pivot = df_pivot.dropna()

        # 4. Cálculo de Deltas: (+) Mejora | (-) Degradación
        df_pivot['delta_pdi'] = df_pivot['durability_pct_Post'] - df_pivot['durability_pct_Pre']
        df_pivot['delta_tons_mes'] = df_pivot['avg_monthly_tons_Post'] - df_pivot['avg_monthly_tons_Pre']
        df_pivot['delta_tph'] = df_pivot['tph_Post'] - df_pivot['tph_Pre']
        df_pivot['delta_load'] = df_pivot['load_hz_Post'] - df_pivot['load_hz_Pre']
        
        self.processed_data = df_pivot
        return self.processed_data

    def run_segmentation(self, n_clusters=4):
        """Etapa de Inteligencia: Clasificación semáforo y clustering causa-raíz."""
        def _get_status(pdi):
            if pdi >= 0: return "1. Mejora o Estable"
            if pdi >= -0.5: return "2. Degradación Leve"
            return "3. Caída"

        self.processed_data['categoria_gerencial'] = self.processed_data['delta_pdi'].apply(_get_status)

        # Clustering sobre variables de proceso para encontrar grupos con comportamiento similar
        features = ['delta_pdi', 'delta_tons_mes', 'delta_tph', 'delta_load']
        X = StandardScaler().fit_transform(self.processed_data[features])
        self.processed_data['cluster_id'] = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(X)
        
        self.results = self.processed_data.reset_index().round(3)
        return self.results

    def get_insights(self):
        """Genera tabla de diagnóstico ejecutivo incluyendo la métrica de carga mecánica."""
        criticos = self.results[self.results['delta_pdi'] < -0.5].copy()
        
        if criticos.empty:
            return None

        def diagnosticar(row):
            causes = []
            if row['delta_tph'] > 0.5: causes.append("Exceso de TPH")
            if row['delta_tons_mes'] > 15: causes.append("Alta Demanda Vol.")
            if row['delta_load'] < -0.5: causes.append("Baja Carga Mecánica")
            return " + ".join(causes) if causes else "Factor Externo"

        criticos['Diagnóstico Causa'] = criticos.apply(diagnosticar, axis=1)
        
        res = criticos[[
            'product', 'delta_pdi', 'delta_tons_mes', 
            'delta_tph', 'delta_load', 'Diagnóstico Causa'
        ]]
        
        res.columns = [
            'Producto', 'Δ PDI (%)', 'Δ Ton/Mes', 
            'Δ TPH', 'Δ Carga (Hz)', 'Diagnóstico Causa'
        ]
        return res

    def plot_comparative_analysis(self):
        """Genera Subplot 1x2 con cuadrantes definidos y estética profesional."""
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=("<b>IMPACTO POR VOLUMEN MENSUAL (TON)</b>", "<b>IMPACTO POR RENDIMIENTO (TON/H)</b>"),
            horizontal_spacing=0.12
        )

        colors = {
            "1. Mejora o Estable": "#27AE60", 
            "2. Degradación Leve": "#F1C40F", 
            "3. Caída": "#E74C3C"
        }
        metrics = [('delta_tons_mes', 'Var. Toneladas Mensuales'), ('delta_tph', 'Var. Rendimiento (Ton/H)')]

        for i, (col_x, label_x) in enumerate(metrics, 1):
            for cat, color in colors.items():
                df_sub = self.results[self.results['categoria_gerencial'] == cat]
                fig.add_trace(go.Scatter(
                    x=df_sub[col_x], y=df_sub['delta_pdi'],
                    mode='markers', name=cat, text=df_sub['product'],
                    hovertemplate="<b>%{text}</b><br>"+label_x+": %{x}<br>Δ PDI: %{y}%<extra></extra>",
                    marker=dict(size=10, color=color, line=dict(width=1, color='white'), opacity=0.9),
                    showlegend=(i == 1)
                ), row=1, col=i)
            
            # Líneas de Cuadrantes (Ejes 0,0)
            fig.add_hline(y=0, line_dash="solid", line_color="#333", opacity=0.6, row=1, col=i)
            fig.add_vline(x=0, line_dash="solid", line_color="#333", opacity=0.6, row=1, col=i)
            
            # Franjas de Salud (Sombreado sutil)
            fig.add_hrect(y0=0, y1=self.results['delta_pdi'].max()*1.2, fillcolor="green", opacity=0.3, layer="below", line_width=0, row=1, col=i)
            fig.add_hrect(y0=self.results['delta_pdi'].min()*1.2, y1=-0.5, fillcolor="red", opacity=0.3, layer="below", line_width=0, row=1, col=i)
            
            fig.update_xaxes(title_text=label_x, row=1, col=i)
            fig.update_yaxes(title_text="Δ PDI % (Calidad)", row=1, col=i)

        main_title = f"<b>MATRIZ DE DESEMPEÑO POR PRODUCTO: {self.pellet_name}</b><br>" \
                     f"<sup>Análisis de Calidad vs. Factores Operativos (1S-2S 2025)</sup>"
        
        fig.update_layout(
            title=main_title, title_x=0.5, template='plotly_white', height=650, width=1250,
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
            margin=dict(t=120)
        )
        return fig


In [112]:

# --- FLUJO DE EJECUCIÓN ---

PELLET_TARGET = "pellet 420"
condiciones = (df_dep["especie"].isin(["Ganadería", "Porcicultura"])) & (df_dep["pellet"] == PELLET_TARGET)

analyzer = QualityPerformanceAnalyzer(df_dep, pellet_name=PELLET_TARGET)
analyzer.transform_data(condiciones)
analyzer.run_segmentation()

# 1. Tabla de Insights (Incluye Delta Load)
reporte_diagnostico = analyzer.get_insights()
if reporte_diagnostico is not None:
    print(f"\n--- DIAGNÓSTICO DE PRODUCTOS CRÍTICOS: {PELLET_TARGET.upper()} ---")
    display(reporte_diagnostico.style.background_gradient(subset=['Δ PDI (%)', 'Δ Carga (Hz)'], cmap='Reds_r').format(precision=3))
    s3.save_dataframe(reporte_diagnostico, f"reporte_diagnostico_{PELLET_TARGET.replace(' ', '_')}.csv")

# 2. Tablero Visual Subplot 1x2
fig_final = analyzer.plot_comparative_analysis()
fig_final.show()

# 3. Guardado Profesional
s3.save_plotly_html(fig_final, f"matriz_estrategica_{PELLET_TARGET.replace(' ', '_')}.html")


--- DIAGNÓSTICO DE PRODUCTOS CRÍTICOS: PELLET 420 ---


,Producto,Δ PDI (%),Δ Ton/Mes,Δ TPH,Δ Carga (Hz),Diagnóstico Causa
7,ALTA ENERGIA R,-1.050,6.000,-0.618,-2.000,Baja Carga Mecánica
8,CERDOS,-0.613,-12.800,0.645,-1.450,Exceso de TPH + Baja Carga Mecánica
10,DESEMPEÑO NORTE PELET,-0.869,-10.333,0.109,-6.956,Baja Carga Mecánica
14,LECHE TOP,-0.564,1.167,-0.306,-8.221,Baja Carga Mecánica
16,LECHONES,-0.683,-7.667,3.037,-3.267,Exceso de TPH + Baja Carga Mecánica
24,PROBIOLECHE,-0.642,-40.914,-0.036,-8.500,Baja Carga Mecánica
25,PROBIOLECHE GRANEL,-0.789,8.597,-0.440,-8.708,Baja Carga Mecánica


In [113]:

# --- FLUJO DE EJECUCIÓN ---

PELLET_TARGET = "pellet 520"
condiciones = (df_dep["especie"].isin(["Ganadería", "Porcicultura"])) & (df_dep["pellet"] == PELLET_TARGET)

analyzer = QualityPerformanceAnalyzer(df_dep, pellet_name=PELLET_TARGET)
analyzer.transform_data(condiciones)
analyzer.run_segmentation()

# 1. Tabla de Insights (Incluye Delta Load)
reporte_diagnostico = analyzer.get_insights()
if reporte_diagnostico is not None:
    print(f"\n--- DIAGNÓSTICO DE PRODUCTOS CRÍTICOS: {PELLET_TARGET.upper()} ---")
    display(reporte_diagnostico.style.background_gradient(subset=['Δ PDI (%)', 'Δ Carga (Hz)'], cmap='Reds_r').format(precision=3))
    s3.save_dataframe(reporte_diagnostico, f"reporte_diagnostico_{PELLET_TARGET.replace(' ', '_')}.csv")
    print( f"reporte_diagnostico_{PELLET_TARGET.replace(' ', '_')}.csv")

# 2. Tablero Visual Subplot 1x2
fig_final = analyzer.plot_comparative_analysis()
fig_final.show()

# 3. Guardado Profesional
s3.save_plotly_html(fig_final, f"matriz_estrategica_{PELLET_TARGET.replace(' ', '_')}.html")
print(f"matriz_estrategica_{PELLET_TARGET.replace(' ', '_')}.html")


--- DIAGNÓSTICO DE PRODUCTOS CRÍTICOS: PELLET 520 ---


,Producto,Δ PDI (%),Δ Ton/Mes,Δ TPH,Δ Carga (Hz),Diagnóstico Causa
17,INICIACION SUPERCERDO,-0.925,4.636,0.263,2.733,Factor Externo
21,LEVANTE BR 454 NUT,-0.900,4.000,-2.044,0.000,Factor Externo
34,MARRANAS GESTACION EB 454 SCP,-0.633,-0.667,-0.562,2.000,Factor Externo


reporte_diagnostico_pellet_520.csv


matriz_estrategica_pellet_520.html
